# Exercise 006

<a href="https://colab.research.google.com/github/FAIRChemistry/PythonProgramming2025/blob/master/exercises/Exercise006.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Please execute this cell to download the necessary data
!wget https://raw.githubusercontent.com/JR-1991/PythonProgramming2025/master/data/all_sequences.fasta

--2026-05-23 07:50:25--  https://raw.githubusercontent.com/JR-1991/PythonProgramming2025/master/data/all_sequences.fasta
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2940315 (2.8M) [text/plain]
Saving to: ‘all_sequences.fasta’

all_sequences.fasta 100%[===================>]   2.80M  --.-KB/s    in 0.07s   

2026-05-23 07:50:25 (42.4 MB/s) - ‘all_sequences.fasta’ saved [2940315/2940315]



# DNASequence class

Read the FASTA file `all_sequences.fasta` and store header info and sequence in a suitable class. Make sure that at the initialization of the object, the following atrributes are present:

* `id`
* `organism`
* `sequence`
* `gc_content`
* `length`

**Tips**

> * Your `__init__`-method arguments do not have to contain all expected attributes if you can derive them from another attribute. The `__init__`-method is a function and you can execute any code you want upon initialization. Make sure to assign your calculation to the appropriate attribute via `self.xyz`.
> * [Dataclasses](https://docs.python.org/3/library/dataclasses.html) are a convinient way to create classes that simply hold data. You can make use of them to simplify the process due to the automatic generation of a `__init__`-method. But keep in mind that this excludes additional calculation you would have otherwise put into your custom `__init__`-method.

In [3]:
# Install biopython automatically for this cell session
!pip install -q biopython

from Bio import SeqIO

class DNASequence:
    def __init__(self, header: str, sequence: str):
        # Save raw inputs
        self.sequence = sequence.upper()

        # Parse the header. Example: "Ecoli_ID1_SequenceHeaderInfo"
        # Split by underscore to isolate the parts
        parts = header.split("_")

        if len(parts) >= 2:
            self.organism = parts[0]
            self.id = parts[1]
        else:
            self.organism = "Unknown"
            self.id = header

        # Automatically calculate biological attributes on creation
        self.length = len(self.sequence)

        # Calculate GC Content ratio
        g_count = self.sequence.count("G")
        c_count = self.sequence.count("C")
        self.gc_content = (g_count + c_count) / self.length if self.length > 0 else 0.0

# Parse the FASTA file and load them into a list of DNASequence instances
dna_objects = []
for record in SeqIO.parse("all_sequences.fasta", "fasta"):
    dna_obj = DNASequence(header=record.description, sequence=str(record.seq))
    dna_objects.append(dna_obj)

# Print a test readout to make sure attributes exist
sample = dna_objects[0]
print(f"Loaded {len(dna_objects)} sequences.")
print(f"Sample - ID: {sample.id} | Organism: {sample.organism} | Length: {sample.length} bp | GC: {sample.gc_content*100:.2f}%")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 27.2 MB/s eta 0:00:00
Loaded 3000 sequences.
Sample - ID: ecoli|1 | Organism: Unknown | Length: 867 bp | GC: 50.75%


## Magic Methods - Alignment by `==`

Can you extend the class to output the identity between the two sequences (stored as an attribute) when the `==` comparison operator is used? Apply the implementation to two sequences that you have chosen and use the supplied `get_identity` function.

Learn more about [Magic methods](https://realpython.com/python-magic-methods/)

In [4]:
# Execute this cell to install all necessary packages
%pip install biopython

In [5]:
# Execute this cell to use the alignment function
from Bio import pairwise2


def get_identity(seq1: str, seq2: str):
    """Aligns two sequences using BioPython

    Args:
        seq1 (str): Query sequence to align to
        seq2 (str): Target sequence to align with

    Returns:
        float: Identity of the resulting alignment

    """
    return pairwise2.align.globalxx(seq1, seq2, score_only=True) / len(seq1)

/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [7]:
# Install biopython automatically for this cell session
!pip install -q biopython

import warnings
from Bio import pairwise2

# Suppress Biopython deprecation warnings to keep output tidy
warnings.filterwarnings("ignore", category=UserWarning)

def get_identity(seq1: str, seq2: str):
    """Aligns two sequences using BioPython"""
    return pairwise2.align.globalxx(seq1, seq2, score_only=True) / len(seq1)


class AlignableDNASequence:
    def __init__(self, header: str, sequence: str):
        self.sequence = sequence.upper()
        self.id = header
        self.length = len(self.sequence)
        self.gc_content = (self.sequence.count("G") + self.sequence.count("C")) / self.length

    # The __eq__ magic method intercepts the '==' operator
    def __eq__(self, other):
        if not isinstance(other, AlignableDNASequence):
            return False  # Can't compare with a completely different object type

        # Calculate alignment identity score using the provided tool
        identity = get_identity(self.sequence, other.sequence)

        print(f"Alignment Identity between {self.id} and {other.id}: {identity * 100:.2f}%")

        # Standard convention: return True if perfectly identical, otherwise False
        return identity == 1.0


# --- Let's test the implementation ---

# Create two mock instances to run an alignment test
seq_a = AlignableDNASequence("Gene_A", "ATGCGTAC")
seq_b = AlignableDNASequence("Gene_B", "ATGCGTAT")  # One nucleotide difference

# Trigger the magic __eq__ method using standard '==' syntax
are_identical = (seq_a == seq_b)
print(f"Are the sequences perfectly identical? {are_identical}")

Alignment Identity between Gene_A and Gene_B: 87.50%
Are the sequences perfectly identical? False
